# 🤖 Classification Supervisée Jour/Nuit - Modèle ML

## Objectif
Développer un algorithme de prédiction pour classifier automatiquement les trains en "jour" ou "nuit".

## Compétences évaluées
- Maîtriser les méthodes et outils de traitement statistique
- Développer des algorithmes de prédiction et suivre leurs performances
- Représenter graphiquement les données

---

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ Imports OK")

---
## 1. Chargement et Exploration des Données

In [ ]:
# Chargement des données
df = pd.read_csv(
    '/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/data/processed/OBRAIL_COMPLETE_FINAL.csv',
    low_memory=False
)

print(f"📊 Dataset: {len(df):,} trains")
print(f"📊 Colonnes: {len(df.columns)}")
print(f"\nClassification actuelle:")
print(df['day_night_classification'].value_counts())
print(f"\nSources de données:")
print(df['data_source'].value_counts())

In [ ]:
# Analyse de la distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution par classification
ax1 = axes[0]
df['day_night_classification'].value_counts().plot(
    kind='bar', ax=ax1, color=['#3498db', '#9b59b6'], rot=0
)
ax1.set_title('Distribution Jour/Nuit (Classification métier)')
ax1.set_xlabel('Type')
ax1.set_ylabel('Nombre de trains')
for i, v in enumerate(df['day_night_classification'].value_counts().values):
    ax1.text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Distribution par source
ax2 = axes[1]
source_order = df['data_source'].value_counts()
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(source_order)))
source_order.plot(kind='barh', ax=ax2, color=colors)
ax2.set_title('Distribution par Source de Données')
ax2.set_xlabel('Nombre de trains')

plt.tight_layout()
plt.savefig('/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/fig_distribution.png', dpi=150)
plt.show()
print("✅ Figure sauvegardée: fig_distribution.png")

---
## 2. Feature Engineering

In [ ]:
def extract_hour(time_str):
    """Extrait l'heure depuis une chaîne HH:MM:SS"""
    try:
        if pd.isna(time_str):
            return None
        parts = str(time_str).split(':')
        return int(parts[0])
    except:
        return None

# Extraction des features
df['heure_depart'] = df['departure_time'].apply(extract_hour)
df['heure_arrivee'] = df['arrival_time'].apply(extract_hour)

# Features cycliques (sinus/cosinus pour capturer la nature cyclique du temps)
df['heure_depart_sin'] = np.sin(2 * np.pi * df['heure_depart'] / 24)
df['heure_depart_cos'] = np.cos(2 * np.pi * df['heure_depart'] / 24)
df['heure_arrivee_sin'] = np.sin(2 * np.pi * df['heure_arrivee'] / 24)
df['heure_arrivee_cos'] = np.cos(2 * np.pi * df['heure_arrivee'] / 24)

# Indicateurs binaires
df['depart_nuit'] = ((df['heure_depart'] >= 22) | (df['heure_depart'] < 6)).astype(int)
df['arrivee_nuit'] = ((df['heure_arrivee'] >= 22) | (df['heure_arrivee'] < 6)).astype(int)

print("✅ Features créées:")
print(f"  - heure_depart, heure_arrivee")
print(f"  - heure_depart_sin, heure_depart_cos (cycliques)")
print(f"  - heure_arrivee_sin, heure_arrivee_cos (cycliques)")
print(f"  - depart_nuit, arrivee_nuit (binaires)")
print(f"\nDataset étendu: {df.shape}")

In [ ]:
# Visualisation des features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Heure de départ par type
ax1 = axes[0, 0]
for label, color in [('day', '#3498db'), ('night', '#9b59b6')]:
    subset = df[df['day_night_classification'] == label]['heure_depart'].dropna()
    ax1.hist(subset, bins=24, alpha=0.6, label=label, color=color)
ax1.set_title('Distribution des heures de départ')
ax1.set_xlabel('Heure')
ax1.set_ylabel('Fréquence')
ax1.legend()
ax1.axvspan(22, 24, alpha=0.2, color='purple', label='Nuit')
ax1.axvspan(0, 6, alpha=0.2, color='purple')

# Durée par type
ax2 = axes[0, 1]
df.boxplot(column='duration_hours', by='day_night_classification', ax=ax2)
ax2.set_title('Durée par type de train')
ax2.set_xlabel('Type')
ax2.set_ylabel('Durée (heures)')
plt.suptitle('')

# Distance par type
ax3 = axes[1, 0]
df.boxplot(column='distance_km', by='day_night_classification', ax=ax3)
ax3.set_title('Distance par type de train')
ax3.set_xlabel('Type')
ax3.set_ylabel('Distance (km)')
plt.suptitle('')

# % nuit par type
ax4 = axes[1, 1]
df.boxplot(column='night_percentage', by='day_night_classification', ax=ax4)
ax4.set_title('Pourcentage de nuit par type')
ax4.set_xlabel('Type')
ax4.set_ylabel('% de nuit')
plt.suptitle('')

plt.tight_layout()
plt.savefig('/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/fig_features.png', dpi=150)
plt.show()
print("✅ Figure sauvegardée: fig_features.png")

---
## 3. Préparation des données pour le ML

In [ ]:
# Sélection des features pour le modèle
feature_cols = [
    'heure_depart', 'heure_arrivee',
    'duration_hours', 'distance_km',
    'night_percentage',
    'heure_depart_sin', 'heure_depart_cos',
    'heure_arrivee_sin', 'heure_arrivee_cos',
    'depart_nuit', 'arrivee_nuit',
    'has_couchette', 'has_sleeper'
]

# Création du dataset ML (suppression des NaN)
df_ml = df[feature_cols + ['day_night_classification', 'data_source']].dropna()
print(f"📊 Dataset ML: {len(df_ml):,} trains (après suppression NaN)")

# Encodage de la target
le = LabelEncoder()
df_ml['target'] = le.fit_transform(df_ml['day_night_classification'])
print(f"\nTarget encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"  day=0, night=1")

# Matrice X et vecteur y
X = df_ml[feature_cols]
y = df_ml['target']

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nDistribution target:")
print(y.value_counts())

In [ ]:
# Corrélation des features
plt.figure(figsize=(12, 10))
corr_matrix = df_ml[feature_cols + ['target']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f')
plt.title('Matrice de Corrélation des Features')
plt.tight_layout()
plt.savefig('/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/fig_correlation.png', dpi=150)
plt.show()
print("✅ Figure sauvegardée: fig_correlation.png")

In [ ]:
# Split Train/Test (stratifié pour préserver la distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Train set: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"📊 Test set:  {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nDistribution Train:")
print(y_train.value_counts(normalize=True))
print(f"\nDistribution Test:")
print(y_test.value_counts(normalize=True))

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"\n✅ Données standardisées")

---
## 4. Entraînement des Modèles

In [ ]:
# Définition des modèles
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100, 
        max_depth=10, 
        random_state=42,
        n_jobs=-1
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    'KNN': KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    )
}

# Dictionnaire pour stocker les résultats
results = {}

print("🚀 Entraînement des modèles...\n")

In [ ]:
# Entraînement et évaluation
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"📊 {name}")
    print('='*50)
    
    # Entraînement
    if name in ['Logistic Regression', 'KNN']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Métriques
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Cross-validation
    if name in ['Logistic Regression', 'KNN']:
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='f1')
    else:
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    
    # Stockage
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    # Affichage
    print(f"\n📈 Métriques sur Test Set:")
    print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"\n📈 Cross-Validation (5-fold):")
    print(f"  F1 Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
    
    # Rapport détaillé
    print(f"\n📋 Rapport de Classification:")
    print(classification_report(y_test, y_pred, target_names=['day', 'night']))

print("\n✅ Entraînement terminé")

---
## 5. Comparaison des Modèles

In [ ]:
# Tableau comparatif
comparison_df = pd.DataFrame({
    name: {
        'Accuracy': res['accuracy'],
        'Precision': res['precision'],
        'Recall': res['recall'],
        'F1-Score': res['f1'],
        'CV F1 Mean': res['cv_mean'],
        'CV F1 Std': res['cv_std']
    }
    for name, res in results.items()
}).T

print("📊 Comparaison des Modèles:\n")
print(comparison_df.round(4))

# Meilleur modèle
best_model = comparison_df['F1-Score'].idxmax()
print(f"\n🏆 Meilleur modèle: {best_model} (F1 = {comparison_df.loc[best_model, 'F1-Score']:.4f})")

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot des métriques
ax1 = axes[0]
comparison_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(
    kind='bar', ax=ax1, colormap='viridis'
)
ax1.set_title('Comparaison des Métriques par Modèle')
ax1.set_xlabel('Modèle')
ax1.set_ylabel('Score')
ax1.set_ylim(0.9, 1.0)
ax1.legend(loc='lower right')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)

# Barplot CV
ax2 = axes[1]
x_pos = np.arange(len(comparison_df))
ax2.bar(x_pos, comparison_df['CV F1 Mean'], 
        yerr=comparison_df['CV F1 Std'], 
        capsize=5, color=['#3498db', '#2ecc71', '#e74c3c'], 
        alpha=0.8)
ax2.set_title('F1-Score en Cross-Validation (5-fold)')
ax2.set_xlabel('Modèle')
ax2.set_ylabel('F1-Score')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(comparison_df.index)
ax2.set_ylim(0.9, 1.0)

plt.tight_layout()
plt.savefig('/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/fig_comparison.png', dpi=150)
plt.show()
print("✅ Figure sauvegardée: fig_comparison.png")

---
## 6. Matrices de Confusion

In [ ]:
# Matrices de confusion pour chaque modèle
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    
    # Normalisation
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues', ax=ax,
                xticklabels=['day', 'night'], yticklabels=['day', 'night'])
    ax.set_title(f'{name}\nF1={res["f1"]:.4f}')
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

plt.tight_layout()
plt.savefig('/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/fig_confusion_matrix.png', dpi=150)
plt.show()
print("✅ Figure sauvegardée: fig_confusion_matrix.png")

In [ ]:
# Matrice de confusion détaillée du meilleur modèle
best_result = results[best_model]

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, best_result['y_pred'])

# Affichage avec counts et pourcentages
group_names = ['Vrai Négatif\n(day → day)', 'Faux Positif\n(day → night)',
               'Faux Négatif\n(night → day)', 'Vrai Positif\n(night → night)']
group_counts = [f'{v:,}' for v in cm.flatten()]
group_percentages = [f'{p:.1%}' for p in cm.flatten()/cm.sum()]

labels = [f'{name}\n{count}\n{pct}' for name, count, pct in 
          zip(group_names, group_counts, group_percentages)]
labels = np.array(labels).reshape(2, 2)

sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', ax=ax,
            xticklabels=['Prédit: day', 'Prédit: night'],
            yticklabels=['Réel: day', 'Réel: night'])
ax.set_title(f'Matrice de Confusion - {best_model}\n(Effectifs et Pourcentages)', fontsize=14)

plt.tight_layout()
plt.savefig('/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/fig_confusion_detailed.png', dpi=150)
plt.show()
print("✅ Figure sauvegardée: fig_confusion_detailed.png")

---
## 7. Importance des Features (Random Forest)

In [ ]:
# Importance des features pour Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(feature_importance['feature'], feature_importance['importance'], 
                color=plt.cm.viridis(feature_importance['importance'] / feature_importance['importance'].max()))
plt.xlabel('Importance')
plt.title('Importance des Features - Random Forest', fontsize=14)
plt.tight_layout()
plt.savefig('/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/fig_feature_importance.png', dpi=150)
plt.show()

print("\n📊 Top 5 Features les plus importantes:")
for i, row in feature_importance.tail(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")
print("\n✅ Figure sauvegardée: fig_feature_importance.png")

---
## 8. Courbe ROC

In [ ]:
# Courbes ROC pour tous les modèles
plt.figure(figsize=(10, 8))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_pred_proba'])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de Faux Positifs (1 - Specificity)')
plt.ylabel('Taux de Vrais Positifs (Recall)')
plt.title('Courbes ROC Comparatives', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/fig_roc_curve.png', dpi=150)
plt.show()
print("✅ Figure sauvegardée: fig_roc_curve.png")

---
## 9. Validation sur Back-on-Track (Vérité Terrain)

In [ ]:
# Back-on-Track contient les vrais trains de nuit (avec couchettes)
df_back_on_track = df_ml[df_ml['data_source'] == 'back_on_track'].copy()

print(f"📊 Trains Back-on-Track: {len(df_back_on_track)}")
print(f"   Tous sont des trains de nuit (ground truth)")

# Prédictions du meilleur modèle
X_bot = df_back_on_track[feature_cols]

if best_model in ['Logistic Regression', 'KNN']:
    X_bot_scaled = scaler.transform(X_bot)
    y_pred_bot = results[best_model]['model'].predict(X_bot_scaled)
else:
    y_pred_bot = results[best_model]['model'].predict(X_bot)

# Analyse
accuracy_bot = (y_pred_bot == 1).mean()  # Tous devraient être night (1)

print(f"\n🎯 Performance sur Back-on-Track (trains de nuit confirmés):")
print(f"   Trains correctement classés 'night': {(y_pred_bot == 1).sum()}/{len(y_pred_bot)}")
print(f"   Trains incorrectement classés 'day': {(y_pred_bot == 0).sum()}/{len(y_pred_bot)}")
print(f"   Accuracy: {accuracy_bot:.2%}")

In [ ]:
# Analyse des erreurs sur Back-on-Track
if (y_pred_bot == 0).sum() > 0:
    print("\n❌ Trains Back-on-Track mal classés:")
    errors_idx = df_back_on_track[y_pred_bot == 0].index
    errors = df.loc[errors_idx][['train_number', 'departure_time', 'arrival_time', 
                                 'duration_hours', 'night_percentage', 'has_couchette', 'has_sleeper']]
    print(errors.to_string())
else:
    print("\n✅ Tous les trains Back-on-Track sont correctement classés!")

---
## 10. Sauvegarde du Modèle

In [ ]:
import joblib

# Sauvegarde du meilleur modèle et du scaler
model_path = '/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/model_jour_nuit.joblib'
scaler_path = '/Users/soufianehamzaoui/Desktop/EPSI/MSPR/1/obrail-mspr/etl/analysis/scaler.joblib'

joblib.dump(results[best_model]['model'], model_path)
joblib.dump(scaler, scaler_path)

print(f"✅ Modèle sauvegardé: {model_path}")
print(f"✅ Scaler sauvegardé: {scaler_path}")

# Test de rechargement
loaded_model = joblib.load(model_path)
print(f"\n✅ Test de rechargement OK")

---
## 11. Synthèse et Conclusions

In [ ]:
print("="*70)
print("📊 SYNTHÈSE - CLASSIFICATION JOUR/NUIT")
print("="*70)

print("\n1️⃣ FEATURES UTILISÉES:")
for i, col in enumerate(feature_cols, 1):
    print(f"   {i}. {col}")

print(f"\n2️⃣ MEILLEUR MODÈLE: {best_model}")
print(f"   - Accuracy:  {results[best_model]['accuracy']:.2%}")
print(f"   - Precision: {results[best_model]['precision']:.2%}")
print(f"   - Recall:    {results[best_model]['recall']:.2%}")
print(f"   - F1-Score: {results[best_model]['f1']:.2%}")

print(f"\n3️⃣ FEATURES LES PLUS IMPORTANTES:")
top_features = feature_importance.tail(3)['feature'].tolist()[::-1]
for i, feat in enumerate(top_features, 1):
    imp = feature_importance[feature_importance['feature'] == feat]['importance'].values[0]
    print(f"   {i}. {feat}: {imp:.2%}")

print(f"\n4️⃣ VALIDATION BACK-ON-TRACK:")
print(f"   - {accuracy_bot:.2%} des trains de nuit confirmés correctement classés")

print("\n5️⃣ LIVRABLES:")
print("   - Notebook ML (ce fichier)")
print("   - Modèle entraîné: model_jour_nuit.joblib")
print("   - Figures: fig_*.png (7 figures)")

print("\n" + "="*70)
print("✅ ANALYSE TERMINÉE")
print("="*70)

---
## 12. Intégration Possible dans l'ETL

```python
# Exemple d'intégration dans etl/transformers/ml_classifier.py

import joblib
import numpy as np

class MLClassifier:
    def __init__(self, model_path, scaler_path):
        self.model = joblib.load(model_path)
        self.scaler = joblib.load(scaler_path)
    
    def predict(self, features_dict):
        # features_dict: {heure_depart, heure_arrivee, duration_hours, ...}
        X = self._prepare_features(features_dict)
        X_scaled = self.scaler.transform(X)
        prediction = self.model.predict(X_scaled)
        probability = self.model.predict_proba(X_scaled)
        return 'night' if prediction[0] == 1 else 'day', probability[0]
```

**Avantages du modèle ML vs règle métier:**
- Prend en compte plus de features (durée, distance, couchettes)
- Score de confiance probabiliste
- Performance validée par métriques quantifiables